# Experiment 005: No Liquidity Filter

**Hypothesis (Mikko):** The static $500k liquidity filter is redundant — the GMX trading universe is small enough that we don't need it. The OI cap already adjusts position sizes dynamically. Let the OI cap and volume sorting do the weighting.

| Arm | Description | Pairlist | Position Sizing |
|-----|-------------|----------|-----------------|
| **A** | Baseline (best from 003/004) | Vol top 75 + Liq $500k | mcap sizing + OI cap + pool cap |
| **B** | No liq filter | Vol top 75 | mcap sizing + OI cap only |
| **C** | Wider universe, no liq filter | Vol top 100 | mcap sizing + OI cap only |
| **D** | Full universe, no filters | All 107 pairs | mcap sizing + OI cap only |

**Strategy:** IchiV3_LS_Static_WhaleCap | **Capital:** $100,000 | **Timerange:** 2021-01-06 to 2026-03-12

**Key difference from Exp 003 Arm E:** This experiment uses the improved WhaleCap strategy from Exp 004 with dust filters (`min_stake_ratio=0.10`, `min_position_pct=0.01`) that prevent slot waste on tiny positions.

In [ ]:
import os, json, zipfile, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio

nest_asyncio.apply()
warnings.filterwarnings('ignore')

PROJECT_ROOT = Path('/home/ubuntu/dev/gmx-ccxt-freqtrade')
os.chdir(PROJECT_ROOT)

STARTING_BALANCE = 100_000
TEMPLATE = 'plotly_dark'

ARM_COLORS = ['#636EFA', '#00CC96', '#AB63FA', '#FFA15A']
ARM_LABELS = ['Arm A: Vol75+Liq+OI (baseline)', 'Arm B: Vol75+OI only', 'Arm C: Vol100+OI only', 'Arm D: All pairs+OI only']
ARM_KEYS = ['arm_a', 'arm_b', 'arm_c', 'arm_d']

RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/005-no-liquidity-filter/results'
ZIP_PATHS = {
    'arm_a': RESULTS_DIR / 'arm_a_vol75_liq_oi.zip',
    'arm_b': RESULTS_DIR / 'arm_b_vol75_oi.zip',
    'arm_c': RESULTS_DIR / 'arm_c_vol100_oi.zip',
    'arm_d': RESULTS_DIR / 'arm_d_all_pairs_oi.zip',
}

for k, p in ZIP_PATHS.items():
    print(f'{k}: {p.name} — exists={p.exists()}')

In [ ]:
# --- Load all result sets ---

def load_trades_from_zip(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        json_name = [n for n in z.namelist() if n.endswith('.json') and 'config' not in n and 'meta' not in n][0]
        with z.open(json_name) as f:
            data = json.load(f)
    strategy_name = list(data['strategy'].keys())[0]
    trades_list = data['strategy'][strategy_name]['trades']
    trades = pd.DataFrame(trades_list)
    trades['open_date'] = pd.to_datetime(trades['open_date'])
    trades['close_date'] = pd.to_datetime(trades['close_date'])
    return trades, strategy_name

all_trades = {}
for key, path in ZIP_PATHS.items():
    trades, strategy = load_trades_from_zip(path)
    all_trades[key] = trades
    print(f'{key}: {len(trades)} trades loaded ({strategy})')

In [ ]:
# --- Side-by-Side Metrics Table ---

def calculate_metrics_raw(trades, starting_balance=STARTING_BALANCE):
    ts = trades.sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = starting_balance + ts['cum_profit_abs']
    
    total_profit_abs = ts['profit_abs'].sum()
    total_profit_pct = (total_profit_abs / starting_balance) * 100
    
    days = (ts['close_date'].max() - ts['open_date'].min()).days
    years = days / 365.25
    final_equity = starting_balance + total_profit_abs
    cagr = ((final_equity / starting_balance) ** (1 / years) - 1) * 100 if years > 0 else 0
    
    equity = ts['equity']
    rolling_max = equity.cummax()
    drawdown = (equity - rolling_max) / rolling_max * 100
    max_dd = drawdown.min()
    
    ts['close_day'] = ts['close_date'].dt.date
    daily_pnl = ts.groupby('close_day')['profit_abs'].sum()
    daily_returns = daily_pnl / starting_balance
    
    sharpe = (daily_returns.mean() / daily_returns.std()) * np.sqrt(365) if daily_returns.std() > 0 else 0
    downside = daily_returns[daily_returns < 0]
    sortino = (daily_returns.mean() / downside.std()) * np.sqrt(365) if len(downside) > 0 and downside.std() > 0 else 0
    calmar = abs(cagr / max_dd) if max_dd != 0 else 0
    
    win_rate = (ts['profit_abs'] > 0).mean() * 100
    
    gross_profit = ts.loc[ts['profit_abs'] > 0, 'profit_abs'].sum()
    gross_loss = abs(ts.loc[ts['profit_abs'] < 0, 'profit_abs'].sum())
    profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')
    
    if 'is_short' in ts.columns:
        longs = ts[~ts['is_short']]
        shorts = ts[ts['is_short']]
    else:
        longs = ts[ts['trade_direction'] == 'long']
        shorts = ts[ts['trade_direction'] == 'short']
    
    # Stake size distribution
    avg_stake = ts['stake_amount'].mean()
    median_stake = ts['stake_amount'].median()
    pct_under_1k = (ts['stake_amount'] < 1000).mean() * 100
    pct_under_5k = (ts['stake_amount'] < 5000).mean() * 100
    
    return {
        'Trades': len(ts),
        'Longs / Shorts': f"{len(longs)} / {len(shorts)}",
        'Profit ($)': round(total_profit_abs, 0),
        'Profit (%)': round(total_profit_pct, 1),
        'CAGR (%)': round(cagr, 1),
        'Max DD (%)': round(max_dd, 1),
        'Sharpe': round(sharpe, 2),
        'Sortino': round(sortino, 2),
        'Calmar': round(calmar, 2),
        'Win Rate (%)': round(win_rate, 1),
        'Profit Factor': round(profit_factor, 2),
        'Avg Trade (%)': round(ts['profit_ratio'].mean() * 100, 2),
        'Avg Stake ($)': round(avg_stake, 0),
        'Median Stake ($)': round(median_stake, 0),
        'Trades <$1k (%)': round(pct_under_1k, 1),
        'Trades <$5k (%)': round(pct_under_5k, 1),
        'Long P&L ($)': round(longs['profit_abs'].sum(), 0),
        'Short P&L ($)': round(shorts['profit_abs'].sum(), 0),
    }

metrics_all = {}
for key, label in zip(ARM_KEYS, ARM_LABELS):
    metrics_all[label] = calculate_metrics_raw(all_trades[key])

metrics_df = pd.DataFrame(metrics_all)
metrics_df

In [ ]:
# --- Delta Table vs Arm A Baseline ---

baseline_col = ARM_LABELS[0]
numeric_metrics = metrics_df.loc[metrics_df[baseline_col].apply(lambda x: isinstance(x, (int, float)))]

delta_df = numeric_metrics.copy()
for col in delta_df.columns:
    if col != baseline_col:
        delta_df[col] = delta_df[col] - delta_df[baseline_col]

delta_df[baseline_col] = '(baseline)'

print('Delta vs Arm A Baseline (positive = higher than baseline):')
delta_df

In [ ]:
# --- Overlaid Equity Curves ---

fig = go.Figure()

for key, label, color in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['equity'],
        mode='lines',
        name=label,
        line=dict(color=color, width=2),
    ))

fig.update_layout(
    title='Equity Curves — All Arms',
    xaxis_title='Date',
    yaxis_title='Equity ($)',
    template=TEMPLATE,
    height=600,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01),
)
fig.show()

In [ ]:
# --- Drawdown Comparison ---

fig = go.Figure()

for key, label, color in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['cum_profit_abs'] = ts['profit_abs'].cumsum()
    ts['equity'] = STARTING_BALANCE + ts['cum_profit_abs']
    rolling_max = ts['equity'].cummax()
    ts['drawdown_pct'] = (ts['equity'] - rolling_max) / rolling_max * 100
    fig.add_trace(go.Scatter(
        x=ts['close_date'],
        y=ts['drawdown_pct'],
        mode='lines',
        name=label,
        line=dict(color=color, width=1.5),
    ))

fig.update_layout(
    title='Drawdown Comparison — All Arms',
    xaxis_title='Date',
    yaxis_title='Drawdown (%)',
    template=TEMPLATE,
    height=500,
    legend=dict(yanchor='bottom', y=0.01, xanchor='left', x=0.01),
)
fig.show()

In [ ]:
# --- Stake Size Distribution ---

fig = make_subplots(rows=2, cols=2, subplot_titles=ARM_LABELS)

positions = [(1,1), (1,2), (2,1), (2,2)]
for (key, label, color, pos) in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS, positions):
    stakes = all_trades[key]['stake_amount']
    fig.add_trace(go.Histogram(
        x=stakes,
        nbinsx=50,
        name=label,
        marker_color=color,
        showlegend=False,
    ), row=pos[0], col=pos[1])

fig.update_layout(
    title='Stake Size Distribution — All Arms',
    template=TEMPLATE,
    height=600,
)
fig.show()

In [ ]:
# --- Long vs Short Profit — Grouped by Arm ---

ls_data = []
for key, label in zip(ARM_KEYS, ARM_LABELS):
    t = all_trades[key].copy()
    if 'is_short' in t.columns:
        t['direction'] = t['is_short'].apply(lambda x: 'Short' if x else 'Long')
    else:
        t['direction'] = t['trade_direction'].apply(lambda x: 'Short' if 'short' in str(x).lower() else 'Long')
    for d in ['Long', 'Short']:
        profit = t.loc[t['direction'] == d, 'profit_abs'].sum()
        ls_data.append({'Arm': label, 'Direction': d, 'Profit': profit})

ls_df = pd.DataFrame(ls_data)

fig = go.Figure()
for i, d in enumerate(['Long', 'Short']):
    subset = ls_df[ls_df['Direction'] == d]
    fig.add_trace(go.Bar(
        x=subset['Arm'],
        y=subset['Profit'],
        name=d,
        marker_color='#636EFA' if d == 'Long' else '#EF553B',
        text=subset['Profit'].apply(lambda x: f'${x:,.0f}'),
        textposition='outside',
    ))

fig.update_layout(
    title='Long vs Short Profit — By Arm',
    xaxis_title='Arm',
    yaxis_title='Profit ($)',
    barmode='group',
    template=TEMPLATE,
    height=500,
)
fig.show()

In [ ]:
# --- Monthly Returns Comparison (Heatmap) ---

monthly_data = {}
for key, label in zip(ARM_KEYS, ARM_LABELS):
    ts = all_trades[key].sort_values('close_date').copy()
    ts['month'] = ts['close_date'].dt.to_period('M')
    monthly_pnl = ts.groupby('month')['profit_abs'].sum()
    monthly_ret = (monthly_pnl / STARTING_BALANCE) * 100
    monthly_data[label] = monthly_ret

monthly_df = pd.DataFrame(monthly_data)
monthly_df.index = monthly_df.index.astype(str)
monthly_df = monthly_df.fillna(0)

fig = go.Figure(data=go.Heatmap(
    z=monthly_df.T.values,
    x=monthly_df.index.tolist(),
    y=monthly_df.columns.tolist(),
    colorscale='RdYlGn',
    zmid=0,
    text=np.round(monthly_df.T.values, 1),
    texttemplate='%{text}%',
    textfont=dict(size=8),
))
fig.update_layout(
    title='Monthly Returns (% of Starting Capital) — All Arms',
    xaxis_title='Month',
    template=TEMPLATE,
    height=350,
    xaxis_tickangle=-45,
)
fig.show()

In [ ]:
# --- Parallelism & Capital Efficiency ---

def compute_daily_exposure(trades, starting_balance=STARTING_BALANCE):
    """For each day, compute open trade count and capital deployed as % of realized balance.
    
    Balance = starting_balance + sum of realized P&L from closed trades.
    This avoids the >100% artifact from unrealized losses shrinking the denominator.
    """
    ts = trades.copy()
    ts['open_date'] = pd.to_datetime(ts['open_date'])
    ts['close_date'] = pd.to_datetime(ts['close_date'])

    date_range = pd.date_range(ts['open_date'].min().normalize(),
                               ts['close_date'].max().normalize(), freq='D')

    # Build realized balance series: starting_balance + cumulative closed trade P&L
    closed_pnl = ts.groupby(ts['close_date'].dt.normalize())['profit_abs'].sum()
    cum_realized = closed_pnl.cumsum().reindex(date_range, method='ffill').fillna(0)
    balance_series = starting_balance + cum_realized

    records = []
    for day in date_range:
        open_mask = (ts['open_date'].dt.normalize() <= day) & (ts['close_date'].dt.normalize() >= day)
        open_trades = ts[open_mask]
        n_open = len(open_trades)
        total_deployed = open_trades['stake_amount'].sum()
        balance = balance_series.loc[day]
        pct_deployed = (total_deployed / balance * 100) if balance > 0 else 0
        records.append({'date': day, 'open_trades': n_open, 'deployed_pct': pct_deployed})

    return pd.DataFrame(records)

def hex_to_rgba(hex_color, alpha=0.3):
    h = hex_color.lstrip('#')
    r, g, b = int(h[0:2], 16), int(h[2:4], 16), int(h[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

# Compute for all arms
daily_exposure = {}
for key, label, color in zip(ARM_KEYS, ARM_LABELS, ARM_COLORS):
    daily_exposure[label] = compute_daily_exposure(all_trades[key])
    de = daily_exposure[label]
    print(f'{label}: avg open trades={de["open_trades"].mean():.1f}, '
          f'avg capital deployed={de["deployed_pct"].mean():.0f}%, '
          f'max open trades={de["open_trades"].max()}')

# --- Parallelism subplot ---
fig = make_subplots(rows=len(ARM_LABELS), cols=1, shared_xaxes=True,
                    subplot_titles=ARM_LABELS, vertical_spacing=0.06)

for i, (label, color) in enumerate(zip(ARM_LABELS, ARM_COLORS), 1):
    de = daily_exposure[label]
    fig.add_trace(go.Scatter(
        x=de['date'], y=de['open_trades'], mode='lines',
        name=label, line=dict(color=color, width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(color, 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['open_trades'].mean()
    fig.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                  annotation_text=f'avg={mean_val:.1f}', annotation_font_color='white',
                  row=i, col=1)
    fig.update_yaxes(title_text='Open Trades', range=[0, 12], row=i, col=1)

fig.update_layout(template=TEMPLATE, height=220 * len(ARM_LABELS),
                  title='Parallelism — Concurrent Open Trades')
fig.show()

In [ ]:
# --- Capital Efficiency — % of Balance Deployed Over Time ---
# Balance = starting capital + realized P&L (closed trades only)
# This shows how well the strategy fills up available capital with positions.

fig2 = make_subplots(rows=len(ARM_LABELS), cols=1, shared_xaxes=True,
                     subplot_titles=ARM_LABELS, vertical_spacing=0.06)

for i, (label, color) in enumerate(zip(ARM_LABELS, ARM_COLORS), 1):
    de = daily_exposure[label]
    deployed_smooth = de['deployed_pct'].rolling(7, min_periods=1).mean()
    fig2.add_trace(go.Scatter(
        x=de['date'], y=deployed_smooth, mode='lines',
        name=label, line=dict(color=color, width=1),
        fill='tozeroy', fillcolor=hex_to_rgba(color, 0.3),
        showlegend=False,
    ), row=i, col=1)
    mean_val = de['deployed_pct'].mean()
    fig2.add_hline(y=mean_val, line_dash='dash', line_color='white', line_width=0.5,
                   annotation_text=f'avg={mean_val:.0f}%', annotation_font_color='white',
                   row=i, col=1)
    fig2.update_yaxes(title_text='% of Balance', range=[0, 100], row=i, col=1)

fig2.update_layout(template=TEMPLATE, height=220 * len(ARM_LABELS),
                   title='Capital Efficiency — Stake Deployed as % of Realized Balance (7d smoothed)')
fig2.show()

In [ ]:
# --- Unique Pairs Traded per Arm ---

for key, label in zip(ARM_KEYS, ARM_LABELS):
    pairs = all_trades[key]['pair'].nunique()
    top_pairs = all_trades[key].groupby('pair')['profit_abs'].sum().sort_values(ascending=False).head(10)
    print(f'\n{label}: {pairs} unique pairs traded')
    print(f'  Top 10 by profit:')
    for pair, profit in top_pairs.items():
        count = len(all_trades[key][all_trades[key]['pair'] == pair])
        print(f'    {pair}: ${profit:,.0f} ({count} trades)')

In [ ]:
# --- Cross-Experiment Comparison: Exp 003 Arm D vs Exp 005 Arms ---

EXP003_RESULTS_DIR = PROJECT_ROOT / 'experiments/ichiv3-gmx/003-volume-pairlist-layers/results'
EXP003_ZIP = EXP003_RESULTS_DIR / 'arm_d_volume_liq_whale.zip'

if EXP003_ZIP.exists():
    exp003_d_trades, _ = load_trades_from_zip(EXP003_ZIP)
    
    cross_metrics = {'Exp003 Arm D (old WhaleCap)': calculate_metrics_raw(exp003_d_trades)}
    for key, label in zip(ARM_KEYS, ARM_LABELS):
        cross_metrics[f'Exp005 {label.split(":")[0]}'] = calculate_metrics_raw(all_trades[key])
    
    cross_df = pd.DataFrame(cross_metrics)
    print('Cross-experiment comparison (Exp 003 Arm D vs all Exp 005 arms):')
    cross_df
else:
    print('Exp 003 Arm D result not found')

## Summary

**TODO:** Fill in after running backtests.

Key questions to answer:
1. Does removing the liquidity filter improve or hurt risk-adjusted returns?
2. Does the OI cap + dust filter adequately replace the liquidity filter's role?
3. Is the wider universe (100 or 107 pairs) better than 75 when the OI cap handles sizing?
4. How does stake size distribution change without the liquidity filter?